Pydantic is used to generate the response in structured schema from LLM

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

c:\Users\Krishna\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [21]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str = Field(description="Title of the movie")
    year:int = Field(description="Year of Release")
    revenue:str = Field(description="Total Revenue of the movie")
    director:str = Field(description="Director of the movie")


In [22]:
model = init_chat_model(model="llama-3.1-8b-instant", model_provider="groq")

In [23]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output.invoke("Details of the movie Avatar")

Movie(title='Avatar', year=2009, revenue='2.788 billion', director='James Cameron')

#### Nested structures

In [25]:
class Actors(BaseModel):
    name:str = Field(description="Name of the actor"),
    age:int = Field(description="Age of the actor")

class Movie(BaseModel):
    title:str = Field(description="Title of the movie")
    year:int = Field(description="Year of Release")
    revenue:str = Field(description="Total Revenue of the movie")
    director:str = Field(description="Director of the movie")
    actors:list[Actors] = Field(description="List of actors in the movie")

In [27]:
nested_model = model.with_structured_output(Movie)
nested_model.invoke("Details of the movie Avatar with Actors")
# nested_model

c:\Users\Krishna\AppData\Local\Programs\Python\Python314\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='Name of the actor'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


Movie(title='Avatar', year=2009, revenue='$2,787,965,087', director='James Cameron', actors=[Actors(name='Sam Worthington', age=43), Actors(name='Zoe Saldana', age=43), Actors(name='Sigourney Weaver', age=74)])

#### TypedDict
TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [34]:
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    title: Annotated[str, "Title of the movie"]
    year: Annotated[int, "Year of Release"]
    revenue: Annotated[str, "Total Revenue of the movie"]
    director: Annotated[str, "Director of the movie"]

structured_model = model.with_structured_output(Movie)

result = structured_model.invoke("Details of the movie Avatar")
print(result)

{'description': 'A young paraplegic marine falls in love with a blue-skinned alien on a distant world.', 'director': 'James Cameron', 'name': 'Avatar', 'revenue': '2.788 billion', 'title': 'Avatar', 'year': 2009}


In [35]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 8192,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

# Dataclass in LangChain

A **Dataclass** is a simple Python class used to define the structure of the output returned by an LLM.

## Example

```python
from dataclasses import dataclass

@dataclass
class Movie:
    title: str
    year: int
```

The model returns a **Movie object**.

```python
print(movie.title)
```

---

# Comparison

| Feature | TypedDict | Dataclass | Pydantic |
|---------|-----------|-----------|-----------|
| Output | Dictionary | Object | Object |
| Access | `movie["title"]` | `movie.title` | `movie.title` |
| Validation | ❌ | ❌ | ✅ |
| Type Conversion | ❌ | ❌ | ✅ |
| Best For | Simple dictionary output | Simple object output | Production LangChain apps |

---

## Which one to use?

- **TypedDict** → Simple dictionary output.
- **Dataclass** → Simple Python object.
- **Pydantic** → Best choice for LangChain because it supports validation and structured outputs.